# 04. 네 가지 실패 모드

Poisoning, Distraction, Confusion, Clash는 입력에서 생기는 문제를 서로 다른 관점으로 설명합니다. 실험에서는 자극의 이름과 실제 출력 관찰을 구분합니다.

## 실행 환경 준비

실습 파일은 `labs/part2/ch08-context-engineering/notebooks/04-네-가지-실패-모드.ipynb`입니다. 새 커널에서 시작해 첫 코드 셀로 패키지 버전을 맞춘 뒤, 설명과 코드를 위에서부터 순서대로 실행합니다. `.env`는 현재 작업 폴더의 상위 경로에서도 찾습니다.

In [1]:
# %pip은 현재 노트북 커널에 패키지를 설치하는 Jupyter 전용 명령입니다.
# -q는 설치 로그를 줄이고, == 뒤 숫자는 수업에서 검증한 버전을 고정합니다.
%pip install -q "langchain==1.4.0" "langchain-openai==1.6.0" "python-dotenv==1.2.3" "tiktoken==0.14.0"

Note: you may need to restart the kernel to use updated packages.


## 1. API 연결 준비

In [2]:
# find_dotenv는 현재 폴더부터 상위 폴더를 검색해 .env 경로를 찾습니다.
from dotenv import find_dotenv, load_dotenv
import os

# usecwd=True는 노트북을 실행한 폴더를 검색 시작점으로 지정합니다.
# 키 값은 출력하지 않고 존재 여부만 확인합니다.
env_path = find_dotenv(usecwd=True)
if not env_path:
    raise FileNotFoundError("현재 폴더의 상위 경로에서 .env를 찾지 못했습니다.")
load_dotenv(env_path)
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError(".env에 OPENAI_API_KEY가 없습니다.")
print(".env와 API 키 확인 완료")

.env와 API 키 확인 완료


### 모델 객체를 만듭니다

`.env` 확인이 끝난 뒤 모델 이름과 출력 제한을 지정합니다. 이 셀은 아직 API에 질문을 보내지 않습니다.

In [3]:
# init_chat_model은 provider:model 형식으로 채팅 모델 객체를 만듭니다.
# model은 이후 셀에서 invoke()를 호출할 때 사용합니다.
from langchain.chat_models import init_chat_model

# openai:는 공급자 이름이며, reasoning_effort="none"은 별도 추론 토큰을 사용하지 않는 설정입니다.
# max_tokens=300은 생성할 출력 토큰 수의 상한입니다.
model = init_chat_model(
    "openai:gpt-5.4-mini",
    reasoning_effort="none",
    max_tokens=300,
)
print("LangChain 모델 준비 완료")

LangChain 모델 준비 완료


## 2. 다섯 가지 입력 조건을 만듭니다

기준 입력을 유지하면서 잘못된 메모, 무관한 정보, 비슷한 기관, 상충 규정을 하나씩 추가합니다.

In [4]:
# clean은 비교의 기준이 되는 근거입니다.
# noise는 길이만 늘리는 무관한 문장입니다.
clean = "[E01 p.2] 미르시립도서관은 매주 월요일 휴관입니다."
noise = "회의 안건: 행사 배너와 좌석 배치를 논의했습니다. " * 12
print("기준 문자 수:", len(clean))
print("무관한 문장 문자 수:", len(noise))

기준 문자 수: 32
무관한 문장 문자 수: 348


### 기준값에 한 가지 자극씩 추가합니다

각 조건은 `clean`을 유지하고 혼동 요소 하나만 더합니다. 조건 이름과 실제 입력 내용은 구분해서 읽습니다.

In [5]:
# conditions는 기준 입력과 네 실패 자극을 이름별로 보관합니다.
conditions = {
    "clean": clean,
    "poisoning": f"""{clean}
[미검증 메모] 휴관일은 화요일입니다.""",
    "distraction": f"""{noise}
{clean}""",
    "confusion": f"""{clean}
[미르고등학교 도서실] 화요일 휴관입니다.""",
    "clash": f"""{clean}
[합성 부록] 미르시립도서관은 수요일 휴관입니다.""",
}
print("실험 조건:", list(conditions))

실험 조건: ['clean', 'poisoning', 'distraction', 'confusion', 'clash']


## 3. 같은 질문으로 다섯 조건을 실행합니다

어떤 실패가 나타날지 미리 정하지 않습니다. 조건 이름, 실제 입력 토큰, 응답 원문을 나란히 기록합니다.

In [6]:
# items()는 조건 이름과 근거를 한 쌍씩 반환하며 모든 조건에 같은 질문을 적용합니다.
# results에는 조건별 AIMessage를 저장하고, usage에서 실제 입력 토큰 수를 읽습니다.
results = {}
for name, context in conditions.items():
    messages = [
        ("system", "근거만 사용하고, 충돌하면 확인이 필요하다고 답하세요."),
        ("user", f"""근거:
{context}

질문: 정기 휴관일은 언제인가요?"""),
    ]
    response = model.invoke(messages)
    usage = response.usage_metadata
    if usage is None:
        raise RuntimeError("모델 사용량 메타데이터가 없습니다.")
    results[name] = response
    print(f"\n[{name}] input_tokens={usage['input_tokens']}")
    print(response.text)


[clean] input_tokens=65
정기 휴관일은 **매주 월요일**입니다.



[poisoning] input_tokens=80
근거에 따르면 **정기 휴관일은 월요일**입니다.  
다만 메모에는 **화요일**이라고 되어 있어 **충돌하므로 추가 확인이 필요합니다**.



[distraction] input_tokens=269
미르시립도서관의 정기 휴관일은 **매주 월요일**입니다.



[confusion] input_tokens=80
근거상 정기 휴관일은 **월요일과 화요일**입니다.

- **미르시립도서관**: 매주 **월요일** 휴관
- **미르고등학교 도서실**: **화요일** 휴관

다만 두 근거가 서로 다른 기관을 가리키므로, **어느 곳의 정기 휴관일을 묻는지에 따라 확인이 필요합니다.**



[clash] input_tokens=85
근거상 **월요일**은 정기 휴관일입니다.  
또한 **수요일**도 휴관이라고 되어 있지만, 서로 충돌하는 정보는 아니고 추가 휴관일로 볼 수 있습니다.

따라서 근거만으로는 **월요일과 수요일**이 휴관일입니다.


## 4. 출력에서 관찰한 문자열을 비교합니다

요일과 확인 요청 표현의 등장 여부만 표시합니다. 이 표만으로 답의 완결성을 확정하지 않고 응답 원문을 함께 읽습니다.

In [7]:
# 각 응답에서 요일 문자열과 추가 확인 표현을 차례로 검사합니다.
for name, response in results.items():
    answer = response.text
    observed = []
    for day in ("월요일", "화요일", "수요일"):
        if day in answer:
            observed.append(day)

    if "확인" in answer or "충돌" in answer or "상충" in answer:
        observed.append("추가 확인 표현")

    if not observed:
        observed = ["요일 표현 없음"]
    print(name, observed)

clean ['월요일']
poisoning ['월요일', '화요일', '추가 확인 표현']
distraction ['월요일']
confusion ['월요일', '화요일', '추가 확인 표현']
clash ['월요일', '수요일', '추가 확인 표현']


## 5. 무관 문장 반복 횟수 한 값만 바꿉니다

근거와 질문은 유지하고 반복 횟수만 `12`에서 `24`로 늘립니다. 이 셀은 추가 API 호출 없이 입력 길이 변화만 비교합니다.

In [8]:
# changed_noise는 같은 문장을 24번 연결한 문자열입니다.
changed_noise = "회의 안건: 행사 배너와 좌석 배치를 논의했습니다. " * 24
base_context = noise + clean
changed_context = changed_noise + clean
print("기본 문자 수:", len(base_context))
print("변경 문자 수:", len(changed_context))

기본 문자 수: 380
변경 문자 수: 728


## 정리

실패 자극을 넣었다는 사실만으로 실패가 발생했다고 기록하지 않습니다. 출력에서 관찰한 문장과 입력 근거를 연결합니다.

## 실패 재현: 상충 규정의 시행일을 기록하지 않습니다

두 규정에 시행일이 없으면 어느 규정이 질문 시점에 유효한지 판정할 수 없습니다.

In [9]:
# items는 서로 상충할 수 있는 규정 문자열 목록입니다.
# 시행일 표기가 없는 항목을 발견하면 즉시 ValueError를 발생시킵니다.
def require_versions(items: list[str]) -> None:
    for item in items:
        if "시행일:" not in item:
            raise ValueError("상충 규정의 시행일이 없습니다.")

### 시행일 없는 두 규정을 전달합니다

`clean`과 `clash`는 요일이 다르지만 시행일이 없습니다. 검사 함수가 이 상태를 거부하는지 확인합니다.

In [10]:
# 두 문자열을 목록으로 묶어 한 번에 검사합니다.
try:
    require_versions([conditions["clean"], conditions["clash"]])
except ValueError as error:
    print("관찰한 실패:", error)
else:
    raise AssertionError("시행일 누락 오류가 발생해야 합니다.")

관찰한 실패: 상충 규정의 시행일이 없습니다.


## 수정: 각 규정에 시행일을 기록합니다

시행일·요일·출처를 구조화하고 질문 기준일에 유효한 규정 가운데 가장 최근 규정을 선택합니다.

In [11]:
# versioned는 시행일·요일·출처를 분리한 규정 목록이고 as_of는 질문 기준일입니다.
# eligible에서 기준일까지 시행된 규정만 남긴 뒤 max의 key로 가장 최근 시행일을 고릅니다.
# YYYY-MM-DD처럼 자릿수가 고정된 날짜는 문자열 순서와 날짜 순서가 같습니다. 다른 형식은 날짜 객체로 바꿉니다.
versioned = [
    {"effective_date": "2025-01-01", "weekday": "월요일", "source": "E01 p.2"},
    {"effective_date": "2026-01-01", "weekday": "수요일", "source": "합성 부록"},
]
as_of = "2026-09-08"
eligible = []
for record in versioned:
    if record["effective_date"] <= as_of:
        eligible.append(record)

def effective_date(record: dict[str, str]) -> str:
    return record["effective_date"]

selected_rule = max(eligible, key=effective_date)
assert selected_rule["weekday"] == "수요일"
assert selected_rule["source"] == "합성 부록"
print("수정 후 적용 규정:", selected_rule)

수정 후 적용 규정: {'effective_date': '2026-01-01', 'weekday': '수요일', 'source': '합성 부록'}
